# Opium Context Exploration (30 Sample Books)
This notebook explores the 30 sampled full texts to manually inspect how opium is mentioned in context.

In [1]:
import os
import re
import pandas as pd
import spacy
import ipywidgets as widgets

from IPython.display import display, HTML

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import spacy.cli
    spacy.cli.download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')


In [2]:
samples_dir = 'samples'
keywords = ['opium', 'poppy', 'anodyne', 'laudanum', 'morphine', 'codein', 'heroin',
    'narcotic', 'soporific', 'paregoric', 'dover\u2019s powder', 'chandu',
    'nepenthe', 'dream-gum', 'dream-stick', 'black smoke']
window_size = 100

results = []
if os.path.exists(samples_dir):
    for filename in os.listdir(samples_dir):
        filepath = os.path.join(samples_dir, filename)
        if os.path.isfile(filepath):
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
                text_lower = text.lower()
                for keyword in keywords:
                    pattern = r'\b' + re.escape(keyword) + r'\b'
                    for match in re.finditer(pattern, text_lower):
                        idx = match.start()
                        left_text = text[:idx]
                        right_text = text[idx + len(keyword):]
                        
                        left_words = left_text.split()
                        right_words = right_text.split()
                        
                        context_left = ' '.join(left_words[-window_size:])
                        context_right = ' '.join(right_words[:window_size])
                        
                        # Simple bounding box handling for text window
                        text_start = max(0, idx - 800)
                        text_end = min(len(text), idx + len(keyword) + 800)
                        raw_snippet = text[text_start:text_end] 
                        
                        results.append({
                            'Book_ID': filename,
                            'Keyword': keyword,
                            'Left_Context': context_left,
                            'Right_Context': context_right,
                            'Full_Context': f"{context_left} {text[idx:idx+len(keyword)]} {context_right}"
                        })
                        
df = pd.DataFrame(results)
print(f"Found {len(df)} keyword mentions across the {len(os.listdir(samples_dir))} sample books.")


Found 113 keyword mentions across the 30 sample books.


## Manual Inspection Viewer
Use the slider to browse through the extracted snippets. The target keyword is highlighted.

In [3]:
pd.set_option('display.max_colwidth', None)
def view_snippet(index):
    if len(df) == 0:
        print("No snippets found.")
        return
    row = df.iloc[index]
    html_out = f"""
    <div style='font-family: Georgia, serif; font-size: 16px; line-height: 1.6; max-width: 800px; padding: 20px; border: 1px solid #ccc; border-radius: 5px; background: #f9f9f9;'>
        <h4>Book ID: {row['Book_ID']} | Keyword: <span style='color: dimgrey;'>{row['Keyword'].upper()}</span></h4>
        <hr>
        <p>
            {row['Left_Context']} 
            <span style='background-color: #ffeb3b; font-weight: bold; padding: 0 4px;'>{row['Keyword']}</span> 
            {row['Right_Context']}
        </p>
    </div>
    """
    display(HTML(html_out))

In [4]:
if len(df) > 0:
    slider = widgets.IntSlider(min=0, max=len(df)-1, step=1, description='Snippet:', layout=widgets.Layout(width='800px'))
    widgets.interact(view_snippet, index=slider)

interactive(children=(IntSlider(value=0, description='Snippet:', layout=Layout(width='800px'), max=112), Outpu…

In [5]:
df[['Book_ID', 'Keyword', 'Full_Context']].to_csv('sample_snippets.csv')

In [ ]:
import csv

keys = results[0].keys()

with open('sample_snippets.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(results)

## Entity & POS Exploration
Let's look at what kinds of entities (PEOPLE, ORG, LOC) or adjectives co-occur in these contexts overall.

In [7]:
from collections import Counter
from tqdm.notebook import tqdm

all_entities = []
all_adjectives = []

for text in tqdm(df['Full_Context'], desc="Processing NLP Contexts"):
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ['PERSON', 'ORG', 'GPE', 'LOC', 'FAC', 'PRODUCT']:
            all_entities.append((ent.label_, ent.text.strip()))
    for token in doc:
        if token.pos_ == 'ADJ' and not token.is_stop and token.is_alpha:
            all_adjectives.append(token.lemma_.lower())

ent_counts = Counter(all_entities).most_common(20)
adj_counts = Counter(all_adjectives).most_common(20)

print("\n--- Top 20 Named Entities in Contexts ---")
for e, c in ent_counts:
    print(f"  {e[0]:<7}: {e[1]} ({c})")

print("\n--- Top 20 Adjectives in Contexts ---")
for a, c in adj_counts:
    print(f"  {a:<15}: {c}")


Processing NLP Contexts:   0%|          | 0/113 [00:00<?, ?it/s]


--- Top 20 Named Entities in Contexts ---
  PERSON : Fogg (15)
  PERSON : Abel (13)
  ORG    : Bulstrode (12)
  GPE    : London (10)
  PERSON : Mary (7)
  GPE    : Hong Kong (6)
  PERSON : Lydgate (6)
  GPE    : Richmond (6)
  PRODUCT: Scarecrow (6)
  PERSON : Bulstrode (5)
  ORG    : Lydgate (5)
  ORG    : the Black Smoke (5)
  ORG    : Lion (5)
  ORG    : mental calmness &c. (5)
  PERSON : ariston metron (5)
  ORG    : lessen &c. (5)
  PERSON : lull (5)
  ORG    : &c. (5)
  GPE    : Paris (4)
  PERSON : Aouda (4)

--- Top 20 Adjectives in Contexts ---
  little         : 35
  black          : 31
  poppy          : 22
  bad            : 20
  high           : 19
  good           : 18
  great          : 17
  sober          : 16
  heavy          : 15
  quiet          : 15
  calm           : 14
  smooth         : 14
  large          : 12
  wild           : 12
  able           : 10
  english        : 10
  long           : 10
  well           : 10
  asleep         : 10
  dead           : 10